## Graph Construction
Build clean road network with boundary.

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd

In [ ]:
boundary_gdf = gpd.read_file("../data/processed/hatyai_boundary.geojson")
polygon = boundary_gdf.union_all()

place_name = "Hat Yai District, Songkhla, Thailand"
graph = ox.graph_from_place(place_name, network_type='drive')
graph_filtered = ox.truncate.truncate_graph_polygon(graph, polygon)

In [ ]:
def remove_isolates_nx(G):
    isolates = list(nx.isolates(G))
    if isolates:
        G = G.copy()
        G.remove_nodes_from(isolates)
    return G

def keep_largest_component_nx(G):
    if G.number_of_nodes() == 0:
        return G
    G_undirected = G.to_undirected()
    largest_nodes = max(nx.connected_components(G_undirected), key=len)
    return G.subgraph(largest_nodes).copy()

graph_result = remove_isolates_nx(graph_filtered)
graph_result = keep_largest_component_nx(graph_result)

In [ ]:
ox.save_graphml(graph_result, "../data/processed/hatyai_graph_clean.graphml")

In [ ]:
print(f"Nodes after cleanup: {graph_result.number_of_nodes()} | edges: {graph_result.number_of_edges()}")
fig, ax = ox.plot_graph(graph_result, figsize=(12, 10))